In [ ]:
# %% Deep learning - Section 23.212
#    CNN GAN with FMNIST

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% FMNIST data

# Define transform (ToTensor brings to [0,1], and Normalise to [-1,1])
transform = T.Compose([ T.Resize((64,64)),
                        T.ToTensor(),
                        T.Normalize(.5,.5) ])

# Import data and apply the transform
dataset = torchvision.datasets.FashionMNIST(root='./data',train=True,download=True,transform=transform)

# Select classes
print(dataset.classes)
keep_classes = [ 'Trouser','Sneaker', 'Pullover'  ]
# keep_classes = [ 'Trouser','Sneaker', 'Sandal'  ]

class_indices = [dataset.classes.index(c) for c in keep_classes]
mask          = torch.isin(dataset.targets, torch.tensor(class_indices))
images2use    = torch.where(mask)[0]

# Convert into DataLoader objects
batch_size  = 100
sampler     = torch.utils.data.SubsetRandomSampler(images2use)
data_loader = DataLoader(dataset,sampler=sampler,batch_size=batch_size,drop_last=True)


In [9]:
# %% Discriminator class

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(  1, 64, 4, 2, 1, bias=False)
        self.conv2 = nn.Conv2d( 64,128, 4, 2, 1, bias=False)
        self.conv3 = nn.Conv2d(128,256, 4, 2, 1, bias=False)
        self.conv4 = nn.Conv2d(256,512, 4, 2, 1, bias=False)
        self.conv5 = nn.Conv2d(512,  1, 4, 1, 0, bias=False)

        # Batchnorm
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self,x):

        x = F.leaky_relu( self.conv1(x) ,.2)

        x = F.leaky_relu( self.conv2(x) ,.2)
        x = self.bn2(x)

        x = F.leaky_relu( self.conv3(x) ,.2)
        x = self.bn3(x)

        x = F.leaky_relu( self.conv4(x) ,.2)
        x = self.bn4(x)

        x = torch.sigmoid( self.conv5(x) ).view(-1,1)

        return x


In [10]:
# %% Generator class

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.ConvTranspose2d(100,512, 4, 1, 0, bias=False)
        self.conv2 = nn.ConvTranspose2d(512,256, 4, 2, 1, bias=False)
        self.conv3 = nn.ConvTranspose2d(256,128, 4, 2, 1, bias=False)
        self.conv4 = nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False)
        self.conv5 = nn.ConvTranspose2d(64,   1, 4, 2, 1, bias=False)

        # Batchnorm
        self.bn1 = nn.BatchNorm2d(512)
        self.bn2 = nn.BatchNorm2d(256)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d( 64)

    def forward(self,x):

        x = F.relu( self.bn1(self.conv1(x)) )
        x = F.relu( self.bn2(self.conv2(x)) )
        x = F.relu( self.bn3(self.conv3(x)) )
        x = F.relu( self.bn4(self.conv4(x)) )
        x = torch.tanh( self.conv5(x) )

        return x


In [ ]:
# %% Test classes on random data

# Discriminator
d_net = DiscriminatorNet()
y     = d_net(torch.randn(10,1,64,64))
print(y.shape)
print(y), print()

# Generator
g_net = GeneratorNet()
y     = g_net(torch.randn(10,100,1,1))
print(y.shape)
print(y[0,:].detach().squeeze().view(64,64).shape)


In [23]:
# %% Set up the GAN ensemble

# Loss funtion is the same for both training phases
loss_fun = nn.BCELoss()

# Model instances
d_net = DiscriminatorNet().to(device)
g_net = GeneratorNet().to(device)

# Optimizer (same but two instances for the two models; GANs usually have a
# small lr and train slowly; betas are based on the tutorial listed above)
d_optimizer = torch.optim.Adam(d_net.parameters(), lr=.0002, betas=(.5,.999))
g_optimizer = torch.optim.Adam(g_net.parameters(), lr=.0002, betas=(.5,.999))

d_optimizer = torch.optim.Adam(d_net.parameters(), lr=.0002)
g_optimizer = torch.optim.Adam(g_net.parameters(), lr=.0002)


In [ ]:
# %% Train the GAN

# Takes ~7 mins on GPU with 1500 batches
num_epochs = int(2500/len(data_loader))

# Preallocate
losses    = []
decisions = []

# Loop
for epoch_i in range(num_epochs):

    for data,_ in data_loader:

        # Ship data to GPU
        data = data.to(device)

        # Labels for real and fake images
        real_labels = torch.ones(batch_size,1).to(device)
        fake_labels = torch.zeros(batch_size,1).to(device)

        # Train the discriminator

        # Forward propagationa and loss for real images (all labels are 1)
        pred_real   = d_net(data)
        d_loss_real = loss_fun(pred_real,real_labels)

        # Forward propagation and loss for fake images (all lablels are 0)
        fake_data   = torch.randn(batch_size,100,1,1).to(device)
        fake_imgs   = g_net(fake_data)
        pred_fake   = d_net(fake_imgs)
        d_loss_fake = loss_fun(pred_fake,fake_labels)

        # Build combined loss (note as you can scale the loss to make the model more
        # or less sensitive to FP or FN)
        d_loss = 1*d_loss_real + 1*d_loss_fake

        # Backpropagation of discriminator
        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Train the generator

        # Create fake images and compute loss
        fake_imgs  = g_net(torch.randn(batch_size,100,1,1).to(device))
        pred_fake  = d_net(fake_imgs)

        # Get loss and discrimination (pass real labels so that the model minimise
        # the loss between pred_fake and real_labels)
        g_loss = loss_fun(pred_fake,real_labels)

        # Backpropagation of generator
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        # Collect losses and decisions
        losses.append([d_loss.item(),g_loss.item()])

        d1 = torch.mean((pred_real>.5).float()).detach().item()
        d2 = torch.mean((pred_fake>.5).float()).detach().item()
        decisions.append([d1,d2])

    # Status message
    msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
    sys.stdout.write('\r' + msg)

# Convert from list to numpy array
losses    = np.array(losses)
decisions = np.array(decisions)


In [18]:
# %% Functions for 1D smoothing filter

# Improved for edge effects - adaptive window
def smooth_adaptive(x,k):
    smoothed = np.zeros_like(x)
    half_k   = k // 2

    for i in range(len(x)):
        start       = max(0, i-half_k)
        end         = min(len(x), i+half_k + 1)
        smoothed[i] = np.mean(x[start:end])

    return smoothed


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(2*phi*6,6))

ax[0].plot(smooth_adaptive(losses[:,0],25))
ax[0].plot(smooth_adaptive(losses[:,1],25))
ax[0].set_xlabel('Batches')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')
ax[0].legend(['Discrimator','Generator'])
# ax[0].set_xlim([500,900])
# ax[0].set_ylim([0,2.5])

ax[1].plot(losses[200:,0],losses[200:,1],'k.',alpha=.1)
ax[1].set_xlabel('Discriminator loss')
ax[1].set_ylabel('Generator loss')
ax[1].set_title('Disc. Loss vs. Gen. Loss')

ax[2].plot(smooth_adaptive(decisions[:,0],25))
ax[2].plot(smooth_adaptive(decisions[:,1],25))
ax[2].set_xlabel('Epochs')
ax[2].set_ylabel('Probablity ("real")')
ax[2].set_title('Discriminator output')
ax[2].legend(['Real','Fake'])

plt.savefig('figure34_gan_cnn_fmnist.png')
plt.show()
files.download('figure34_gan_cnn_fmnist.png')


In [ ]:
# %% Plotting

# Generate the images with the generator network
g_net.eval()
fake_data = g_net(torch.randn(batch_size,100,1,1).to(device)).cpu()

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,6,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):
    ax.imshow(fake_data[i,:,].detach().squeeze(),cmap='grey')
    ax.axis('off')

plt.suptitle('Generated FMNIST')

plt.savefig('figure35_gan_cnn_fmnist.png')
plt.show()
files.download('figure35_gan_cnn_fmnist.png')


In [ ]:
# %% Exercise 1
#    I've mentioned before that GANs can be quite sensitive to subtle changes in model architecture. Try running the code
#    again with exactly one change: Set the Adam 'betas' parameters to their default values (simply remove that argument
#    from the code. How much of an impact does this have on the results? More generally, these sensitivities can be
#    frustrating when trying to build new models; the best thing to do is search the web for similar kinds of models and
#    be inspired by their decisision (but don't assume that a model is good just because it's on the web!).

# The performance drops quite a bit, the images are now less detailed and a
# checkerboard pattern appears in the background
